# Retrieval evaluation for a RAG pipeline
A small project to learn how the retrieval part of a RAG pipeline works. I index Transformer, BERT and RAG and test how well semantic search finds the right passage. The main question: how does chunk size affect retrieval quality?

# Setup

In [ ]:
!pip install -q pypdf sentence-transformers chromadb

# Data: three papers from arXiv
They are downloaded when the notebook runs, so no PDFs are stored in this repo

In [ ]:
import os, requests

os.makedirs("pdfs", exist_ok=True)
papers = {
    "attention.pdf": "https://arxiv.org/pdf/1706.03762",  # Attention Is All You Need
    "bert.pdf": "https://arxiv.org/pdf/1810.04805",       # BERT
    "rag.pdf": "https://arxiv.org/pdf/2005.11401",        # Retrieval-Augmented Generation
}
headers = {"User-Agent": "Mozilla/5.0 (research-demo)"}
for name, url in papers.items():
    r = requests.get(url, headers=headers, timeout=60)
    r.raise_for_status()
    with open(f"pdfs/{name}", "wb") as f:
        f.write(r.content)
    print(name, round(len(r.content) / 1024), "KB")

attention.pdf 2163 KB
bert.pdf 757 KB
rag.pdf 865 KB


# I split every page into overlapping chunks
Chunk size is the main thing I compare in an evaluation section

In [ ]:
from pypdf import PdfReader

CHUNK_SIZE = 800   # used for the interactive search below (the evaluation tests 400, 800 and 1200 automatically)
OVERLAP = 150

def load_pdf(path):
    reader = PdfReader(path)
    return [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]

def chunk_text(text, size=800, overlap=150):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks

all_pages = []   # (file, page number, text)
for fname in sorted(os.listdir("pdfs")):
    for page_no, text in load_pdf(f"pdfs/{fname}"):
        all_pages.append((fname, page_no, text))

def make_chunks(chunk_size, overlap):
    docs, metas, ids = [], [], []
    for fname, page_no, text in all_pages:
        for j, chunk in enumerate(chunk_text(text, chunk_size, overlap)):
            if len(chunk.strip()) < 50:   # skip nearly empty chunks
                continue
            docs.append(chunk)
            metas.append({"source": fname, "page": page_no})
            ids.append(f"{fname}-{page_no}-{j}")
    return docs, metas, ids

documents, metadatas, ids = make_chunks(CHUNK_SIZE, OVERLAP)
print("Number of chunks:", len(documents))

Number of chunks: 285


# Embeddings and vector database
Each text fragment is transformed into a vector using a lightweight semantic embedding model and stored in ChromaDB

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(documents, show_progress_bar=True).tolist()

db = chromadb.Client()
collection = db.get_or_create_collection("papers")

BATCH = 500
for i in range(0, len(documents), BATCH):
    collection.add(
        documents=documents[i:i + BATCH],
        embeddings=embeddings[i:i + BATCH],
        metadatas=metadatas[i:i + BATCH],
        ids=ids[i:i + BATCH],
    )
print("Stored:", collection.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Stored: 285


# A quick check that search works

In [ ]:
def search(question, k=4):
    q_emb = embedder.encode([question]).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=k)
    return [
        (res["documents"][0][i], res["metadatas"][0][i], res["distances"][0][i])
        for i in range(len(res["documents"][0]))
    ]

def show(question, k=4):
    print("QUESTION:", question, "\n")
    for text, meta, dist in search(question, k):
        print(f"[{meta['source']}, page {meta['page']}] distance={dist:.2f}")
        print(text[:400].replace("\n", " "), "\n---")

show("What is multi-head attention?")

QUESTION: What is multi-head attention? 

[attention.pdf, page 14] distance=0.70
ll attentions for head 5. Bottom: Isolated attentions from just the word ‘its’ for attention heads 5 and 6. Note that the attentions are very sharp for this word. 14 
---
[attention.pdf, page 15] distance=0.81
nce. We give two such examples above, from two different heads from the encoder self-attention at layer 5 of 6. The heads clearly learned to perform different tasks. 15 
---
[attention.pdf, page 5] distance=0.85
o the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality. 3.2.3 Applications of Attention in our Model The Transformer uses multi-head attention in three different ways: • In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the e 
---
[attention.pdf, page 5] distance=0.90
output values. These are concatenated and once again 

## Evaluation: which chunk size works best?

I compare chunk sizes of 400, 800 and 1200 characters on 36 questions. Each question has a known paper and an exact answer phrase.

- **paper@1:** the top result comes from the correct paper (easy test)
- **phrase in top k:** a retrieved chunk really contains the answer (hard test)
- **MRR:** how high the first correct chunk is ranked (searched in the top 10)

To keep the comparison fair, k is chosen so that every chunk size returns about 3600 characters of context. The phrase check ignores case, spaces, line breaks, hyphens and Unicode ligatures


In [ ]:
import re, unicodedata

tests = [
    # (question, paper that contains the answer, exact phrase from that paper)
    ("What is multi-head attention?", "attention.pdf", "jointly attend"),
    ("Why are the dot products scaled in scaled dot-product attention?", "attention.pdf", "extremely small gradients"),
    ("How many layers and what hidden size does BERT-base have?", "bert.pdf", "768"),
    ("What fraction of the tokens is masked in masked language modeling?", "bert.pdf", "15%"),
    ("What architecture does the retriever in RAG follow?", "rag.pdf", "bi-encoder"),
    ("What are the two RAG formulations?", "rag.pdf", "RAG-Token"),
    ("What is the name of the positional representation used in Transformer?", "attention.pdf", "positional encoding"),
    ("What is the secondary pre-training task in BERT besides MLM?", "bert.pdf", "Next Sentence Prediction"),
    ("What pre-trained model is used as the generator in RAG?", "rag.pdf", "BART"),
    ("What is the maximum sequence length used in BERT training?", "bert.pdf", "512"),
    ("What kind of memory does RAG combine?", "rag.pdf", "parametric memory"),
    ("How many parallel attention layers does the base Transformer use in each multi-head block?", "attention.pdf", "parallel attention layers"),
    ("Where is dropout applied inside the Transformer?", "attention.pdf", "to the output of each sub-layer"),
    ("Which optimizer was used to train the Transformer?", "attention.pdf", "Adam optimizer"),
    ("What translation quality did the big Transformer reach on English-to-German?", "attention.pdf", "28.4 BLEU"),
    ("How many layers does the encoder stack have?", "attention.pdf", "stack of N = 6 identical layers"),
    ("Why is self-attention good for long-range dependencies?", "attention.pdf", "path length"),
    ("What does the Transformer drop compared to older sequence models?", "attention.pdf", "dispensing with recurrence"),
    ("How does the decoder avoid looking at future words?", "attention.pdf", "attending to subsequent positions"),
    ("What does the name BERT stand for?", "bert.pdf", "Bidirectional Encoder Representations from Transformers"),
    ("What GLUE score did BERT reach?", "bert.pdf", "80.5%"),
    ("How many parameters does the large BERT model have?", "bert.pdf", "340M"),
    ("Which special token starts every input sequence for classification?", "bert.pdf", "special classification token"),
    ("How are two sentences marked as separate in the BERT input?", "bert.pdf", "separate them with a special token"),
    ("How often is a selected token replaced by the mask token during pre-training?", "bert.pdf", "80% of the time"),
    ("Which text corpora was BERT pre-trained on?", "bert.pdf", "BooksCorpus"),
    ("How are the sentence pairs for next sentence prediction built?", "bert.pdf", "50% of the time"),
    ("How many sequences are in one pre-training batch?", "bert.pdf", "256 sequences"),
    ("What kind of search problem is finding the top documents in RAG?", "rag.pdf", "Maximum Inner Product Search"),
    ("What is RAG's non-parametric memory?", "rag.pdf", "dense vector index of Wikipedia"),
    ("How long are the text passages stored in the document index?", "rag.pdf", "100-word chunks"),
    ("Which document does RAG-Sequence use to generate the whole output?", "rag.pdf", "same retrieved document"),
    ("What can RAG-Token do that RAG-Sequence cannot?", "rag.pdf", "different latent document"),
    ("On which kind of tasks did RAG set the state of the art?", "rag.pdf", "open domain QA tasks"),
    ("What does the abbreviation RAG stand for?", "rag.pdf", "Retrieval-Augmented Generation"),
    ("How is retrieved text given to the generator together with the input?", "rag.pdf", "simply concatenate them"),
]

def norm(s):
    s = unicodedata.normalize("NFKC", s)
    return re.sub(r"[^a-z0-9%]+", "", s.lower())


def build_collection(docs, metas, ids_, name):
    try:
        db.delete_collection(name)
    except Exception:
        pass
    col = db.create_collection(name)
    emb = embedder.encode(docs).tolist()
    for i in range(0, len(docs), 500):
        col.add(documents=docs[i:i + 500], embeddings=emb[i:i + 500],
                metadatas=metas[i:i + 500], ids=ids_[i:i + 500])
    return col

def search_in(col, question, k):
    q = embedder.encode([question]).tolist()
    res = col.query(query_embeddings=q, n_results=k)
    return list(zip(res["documents"][0], res["metadatas"][0]))

COLS = {}

def evaluate(chunk_size, overlap=150, context_chars=3600, kmax=10):
    k = max(1, round(context_chars / chunk_size))
    docs, metas, ids_ = make_chunks(chunk_size, overlap)
    col = build_collection(docs, metas, ids_, f"eval_{chunk_size}")
    COLS[chunk_size] = col
    paper1, phrase_k, rr_sum, missed = 0, 0, 0.0, []
    for question, src, phrase in tests:
        hits = search_in(col, question, kmax)
        ph = norm(phrase)
        if hits[0][1]["source"] == src:
            paper1 += 1
        rank = next((r for r, (t, _) in enumerate(hits, 1) if ph in norm(t)), None)
        if rank:
            rr_sum += 1 / rank
        if rank is not None and rank <= k:
            phrase_k += 1
        else:
            missed.append(question)
    n = len(tests)
    row = {"chunk size": chunk_size, "k": k, "chunks": len(docs),
           "paper@1": f"{paper1}/{n}", "phrase in top k": f"{phrase_k}/{n}", "MRR": round(rr_sum / n, 3)}
    return row, missed

results, all_missed = [], {}
for size in [400, 800, 1200]:
    row, missed = evaluate(size)
    results.append(row)
    all_missed[size] = missed

header = list(results[0].keys())
print(" | ".join(f"{h:>15}" for h in header))
for row in results:
    print(" | ".join(f"{str(row[h]):>15}" for h in header))

     chunk size |               k |          chunks |         paper@1 | phrase in top k |             MRR
            400 |               9 |             706 |           32/36 |           27/36 |           0.364
            800 |               4 |             285 |           36/36 |           23/36 |           0.465
           1200 |               3 |             184 |           33/36 |           15/36 |           0.428


In [ ]:
import pandas as pd
pd.DataFrame(results)


,chunk size,k,chunks,paper@1,phrase in top k,MRR
0,400,9,706,32/36,27/36,0.364
1,800,4,285,36/36,23/36,0.465
2,1200,3,184,33/36,15/36,0.428


# What went wrong: reading the misses

In [ ]:
def inspect(question, size=800, n=3):
    print("QUESTION:", question)
    for text, meta in search_in(COLS[size], question, n):
        print(meta, "|", text[:300].replace("\n", " "))
        print("---")

for q in all_missed[800]:
    inspect(q)
    print("=" * 60)

QUESTION: What is the name of the positional representation used in Transformer?
{'page': 5, 'source': 'attention.pdf'} | o the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality. 3.2.3 Applications of Attention in our Model The Transformer uses multi-head attention in three different ways: • In "encoder-decoder attention" layers, the querie
---
{'page': 3, 'source': 'attention.pdf'} | Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively. 3.1 Encoder and Decoder Stacks Encoder
---
{'source': 'bert.pdf', 'page': 3} | rate ﬁne-tuned models, even though they are ini- tialized with the same pre-trained parameters. The question-answering example in Figure 1 will serve as a running example for this section. A 

# Sanity check: every test phrase occurs in its PDF

In [ ]:
full = {}
for fname, page_no, text in all_pages:
    full[fname] = full.get(fname, "") + " " + text
for q, src, phrase in tests:
    if norm(phrase) not in norm(full[src]):
        print("PHRASE NOT IN PAPER:", phrase, "|", q)
